In [1]:


import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
google_api_key = os.getenv('GEMINI_API_KEY')
gemini_url="https://generativelanguage.googleapis.com/v1beta/openai/"

In [3]:

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [5]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget', 'choco'],
 'cpu': {'brand': 'AMD Ryzen 7 7730U with Radeon Graphics',
  'cores_logical': 16,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 16.1.0',
   'g++': 'g++.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 16.1.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 4.3.2',
   'ninja': '1.13.2',
   'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Great news! You are already set up to compile and run C++ code. Your system information shows that you have `gcc` and `g++` installed, which are part of the GNU Compiler Collection and are very capable C++ compilers. Specifically, you have a MinGW-W64 installation, which is a popular choice for developing on Windows.

Here's how to compile and execute your `main.cpp` file using the simplest approach with the fastest possible runtime performance:

### Compiling and Executing C++ Code

To achieve the fastest possible runtime performance with `g++`, you'll want to enable optimizations. The `-O3` flag tells the compiler to apply a high level of optimization. The `-march=native` flag (or its MinGW equivalent, often implied by `-O3` for performance, but good to be aware of for explicit control) tells the compiler to tune the optimizations for your specific CPU architecture, which can yield significant speedups.

Here are the `compile_command` and `run_command` for your Python script:

```python
import subprocess

# Assuming your C++ file is named main.cpp
cpp_file = "main.cpp"
executable_name = "main.exe" # For Windows, executables typically have a .exe extension

# Compile command for fastest runtime performance
# -O3: Enables aggressive optimizations for speed.
# -o: Specifies the output executable name.
# -std=c++17 or -std=c++20 (optional but recommended): Specifies the C++ standard.
#   Use the highest standard your code supports for modern features and potential optimizations.
compile_command = [
    "g++",
    cpp_file,
    "-O3",          # Aggressive optimizations for speed
    "-o", executable_name,
    "-std=c++17"    # You can change this to -std=c++20 if your code uses C++20 features
]

# Run command
run_command = [f"./{executable_name}"] # The './' is important on Windows to specify the current directory


try:
    print(f"Compiling: {' '.join(compile_command)}")
    compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print("Compilation successful.")
    print("Compiler stdout:\n", compile_result.stdout)
    print("Compiler stderr:\n", compile_result.stderr)

    print(f"\nRunning: {' '.join(run_command)}")
    run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
    print("Execution successful.")
    print("Program stdout:\n", run_result.stdout)
    print("Program stderr:\n", run_result.stderr)

    # Return the stdout of the executed program
    # return run_result.stdout

except subprocess.CalledProcessError as e:
    print(f"Error during compilation or execution:")
    print(f"Command: {e.cmd}")
    print(f"Return code: {e.returncode}")
    print(f"Stdout:\n{e.stdout}")
    print(f"Stderr:\n{e.stderr}")
    # Handle the error as appropriate
except FileNotFoundError:
    print("Error: g++ command not found. Please ensure MinGW-W64 is correctly installed and in your PATH.")

```

### Explanation:

1.  **`compile_command`**:
    *   `"g++"`: This invokes your installed GNU C++ compiler.
    *   `cpp_file`: This is the name of your C++ source file (`main.cpp`).
    *   `"-O3"`: This is the crucial flag for **fastest possible runtime performance**. It enables a high level of compiler optimizations. This will make your compiled executable run as fast as the compiler can make it.
    *   `"-o", executable_name`: This tells `g++` to name the output executable file `main.exe`.
    *   `"-std=c++17"` (or `"-std=c++20"`): This specifies the C++ standard you are using. It's good practice to explicitly state this. `c++17` and `c++20` are the most recent and offer modern language features and optimizations. Use the highest standard your code supports.

2.  **`run_command`**:
    *   `[f"./{executable_name}"]`: On Windows, to run an executable in the current directory, you need to prefix it with `./`. This is because the current directory is not always in the system's `PATH` by default for security reasons.

**To use this:**

1.  Save your C++ code in a file named `main.cpp` in the same directory where you'll run this Python script.
2.  Save the Python code above as a `.py` file (e.g., `compile_run.py`).
3.  Run the Python script from your terminal: `python compile_run.py`

The script will first compile `main.cpp` into `main.exe` with optimizations, and then it will execute `main.exe` and print its output.

In [29]:
compile_command = [
    "g++",
    "main.cpp",
    "-O2",
    "-std=c++17",
    "-o",
    "main.exe"
]

run_command = [".\\main.exe"]

run_command = ["main.exe"]

In [30]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""


In [31]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [32]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [33]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [34]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(1000000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [35]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [36]:
run_python(pi)

Result: 3.141593153589
Execution Time: 0.215337 seconds


In [ ]:
port(gemini, GEMINI_MODEL, pi)

In [ ]:
import subprocess

def compile_and_run():
    compile_result = subprocess.run(
        compile_command,
        text=True,
        capture_output=True
    )

    print("Compile return code:", compile_result.returncode)
    print("Compile stderr:")
    print(compile_result.stderr)

    run_result = subprocess.run(
        run_command,
        text=True,
        capture_output=True
    )

    print("Run return code:", run_result.returncode)
    print("STDOUT:")
    print(run_result.stdout)
    print("STDERR:")
    print(run_result.stderr)

In [ ]:
print(os.getcwd())
print(os.path.exists("main.cpp"))
print(compile_command)


In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

In [38]:
import gradio as gr

d:\MLProjects\GenAI\projects\Week4Projects\LLM-Code-Optimizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]



In [42]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

d:\MLProjects\GenAI\projects\Week4Projects\LLM-Code-Optimizer\.venv\Lib\site-packages\gradio\utils.py:1216: UserWarning: Expected 3 arguments for function <function port at 0x000002546BB06700>, received 2.
  warnings.warn(
d:\MLProjects\GenAI\projects\Week4Projects\LLM-Code-Optimizer\.venv\Lib\site-packages\gradio\utils.py:1220: UserWarning: Expected at least 3 arguments for function <function port at 0x000002546BB06700>, received 2.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
